In [ ]:
import pymysql
conn=pymysql.connect(host='localhost',
                     user='root',
                     password='2255',
                     autocommit=True)

In [2]:
cursor=conn.cursor()

In [3]:
def create_table(date,cursor,database):
    cursor.execute(f'use {database}')
    cursor.execute(f"show tables")
    tables=cursor.fetchall()
    table_names = [t[0] for t in tables]
    if str(date) not in table_names:
        cursor.execute(f"create table `{str(date)}` (name varchar(100),time_of_attend time)")

In [4]:
import datetime
a=datetime.datetime.now().time()

In [5]:
str(a).split(".")[0]

'12:34:00'

In [6]:
def check_and_add(name,date,cursor,time):
    cursor.execute(f'select name from `{str(date)}`')
    students=cursor.fetchall()
    student_name=[s[0] for s in students]
    if name not in student_name:
        cursor.execute(f'insert into `{str(date)}` values (%s,%s)',(name,str(time).split('.')[0]))
        return True
    else:
        return False

In [7]:
data={}

In [8]:
def add_person(name,image_path,data):
    import face_recognition
    try:
        img_array=face_recognition.load_image_file(image_path)
        img_vector=face_recognition.face_encodings(img_array)
        if name in data and img_vector:
            print("The person already exist.")
            vector=img_vector[0]
            data[name]=vector
            print(f"New face updated for the person {name}")
            return data
    
        if img_vector:
            vector=img_vector[0]
            data[name]=vector
            return data
        
    except Exception:
        return None

Main Code

In [9]:
import face_recognition
import cv2

In [10]:
data={}
names=[("Dixson","dixson_luminar.jpg"),
       ("Dulquer","dulq.jpg"),
       ("Mammooty","mammooty.jpg"),
       ("Mohanlal","mohanlal.jpg")]
for i,j in names:
    add_person(i,j,data)    

In [11]:
import pickle
with open('face_vector.pkl','wb') as obj1:
    pickle.dump(data,obj1)

In [12]:
[False,False,False,False]

[False, False, False, False]

In [16]:
cap=cv2.VideoCapture(0)
import datetime
import time
d=datetime.datetime.now().date()
# cursor.execute("create database ml_feb_mar")
create_table(d,cursor,'ml_feb_mar')
while True:
    ret,frame=cap.read()
    if not ret:
        print("No frame")
        break
    tm=datetime.datetime.now().time()
    frame_rgb=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    frame_encodings=face_recognition.face_encodings(frame_rgb)
    h,w,_=frame.shape
    if frame_encodings:
        for unk_vect in frame_encodings:
            result=face_recognition.compare_faces(unk_vect,list(data.values()),tolerance=0.5)# []
            if True in result:
                ind=result.index(True)
                name=list(data.keys())[ind]
                add_status=check_and_add(name,datetime.datetime.now().date(),cursor,datetime.datetime.now().time())
                if add_status:
                    cv2.putText(frame,f"Attendance marked for the student {name}",(10,h-10),6,1,(255,255,255),1)
                    time.sleep(2)
                else:
                    cv2.putText(frame,f"Attendance already marked for student {name}",(10,h-10),6,1,(255,255,255),1)

                
            else:
                cv2.putText(frame,'Unknown face',(10,h-10),6,1,(255,255,255),1)
    else:
        cv2.putText(frame,'No face detected.',(10,h-10),6,1,(255,22,32),1)
    cv2.imshow('face_recognition',frame)
    if cv2.waitKey(10)==ord('k'):
        cv2.destroyAllWindows()
        cap.release()
        break

In [ ]:
import pymysql
import face_recognition
import cv2
import datetime
import time

# MySQL connection
conn = pymysql.connect(host='localhost', user='root', password='12345')
cursor = conn.cursor()

def create_table(date, cursor, database):
    cursor.execute(f'USE {database}')
    cursor.execute("SHOW TABLES")
    table_names = [t[0] for t in cursor.fetchall()]
    if str(date) not in table_names:
        cursor.execute(f"CREATE TABLE `{str(date)}` (name VARCHAR(100), time_of_attend TIME)")

def check_and_add(name, date, cursor, connector, time):
    cursor.execute(f'SELECT name FROM `{str(date)}`')
    students = [s[0] for s in cursor.fetchall()]
    if name not in students:
        cursor.execute(f'INSERT INTO `{str(date)}` (name, time_of_attend) VALUES (%s, %s)', (name, str(time).split('.')[0]))
        connector.commit()
        return True
    else:
        return False

def add_person(name, image_path, data):
    try:
        img_array = face_recognition.load_image_file(image_path)
        img_vector = face_recognition.face_encodings(img_array)
        if img_vector:
            data[name] = img_vector[0]
            return data
    except Exception as e:
        print(f"Error loading {name}'s image: {e}")
        return None

# Load known faces
data = {}
names = [
    ("Dixson", "dixson_luminar.jpg"),
    ("Dulquer", "dulq.jpg"),
    ("Mammooty", "mammooty.jpg"),
    ("Mohanlal", "mohanlal.jpg")
]
for name, img in names:
    add_person(name, img, data)

# Start attendance
cap = cv2.VideoCapture(0)
d = datetime.datetime.now().date()

create_table(d, cursor, 'ml_nov_25')

print("Press 'k' to quit...")

while True:
    ret, frame = cap.read()
    if not ret:
        print("No frame")
        break

    tm = datetime.datetime.now().time()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame_encodings = face_recognition.face_encodings(frame_rgb)
    h, w, _ = frame.shape

    if frame_encodings:
        for unk_vect in frame_encodings:
            result = face_recognition.compare_faces(list(data.values()), unk_vect, tolerance=0.5)
            if True in result:
                ind = result.index(True)
                name = list(data.keys())[ind]
                add_status = check_and_add(name, d, cursor, conn, tm)
                msg = f"Attendance {'marked' if add_status else 'already marked'} for {name}"
            else:
                msg = "Unknown face"
    else:
        msg = "No face detected"

    cv2.putText(frame, msg, (10, h - 10), cv2.FONT_HERSHEY_COMPLEX, 0.5, (0, 255, 0), 1)
    cv2.imshow('Face Recognition', frame)

    if cv2.waitKey(10) == ord('k'):
        break

cap.release()
cv2.destroyAllWindows()


Press 'k' to quit...
